In [21]:
import torch
from torch import nn
import math

In [22]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(EncoderLayer, self).__init__()

        self.self_attn = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, memory_mask=None):
        # Self-attention
        attn_output, _ = self.self_attn(x, x, x, attn_mask=memory_mask)
        x = self.norm1(x + attn_output)  # Add & Norm
        
        # Feed-forward network
        ffn_output = self.ffn(x)
        ffn_output = self.dropout(ffn_output)
        x = self.norm2(x + ffn_output)  # Add & Norm
        
        return x

In [23]:
layer = EncoderLayer(embed_dim=512, num_heads=8, ff_dim=1024)
print(layer)

EncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (ffn): Sequential(
    (0): Linear(in_features=512, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
  )
  (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [24]:
class DecoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(DecoderLayer, self).__init__()

        self.self_attn = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)

        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.norm3 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, enc_output, tgt_mask=None, memory_mask=None): # tgt_mask: Mask cho self-attention, memory_mask: Mask cho cross-attention (mask padding hoặc mask causal)
        # Self-attention
        attn_output, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + attn_output)  # Add & Norm
        
        # Cross-attention
        cross_attn_output, _ = self.cross_attn(x, enc_output, enc_output, attn_mask=memory_mask) # Query từ decoder, Key & Value từ encoder
        x = self.norm2(x + cross_attn_output)  # Add & Norm
        
        # Feed-forward network
        ffn_output = self.ffn(x)
        ffn_output = self.dropout(ffn_output)
        x = self.norm3(x + ffn_output)  # Add & Norm
        
        return x

In [25]:
layer = DecoderLayer(embed_dim=512, num_heads=8, ff_dim=1024)
print(layer)

DecoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (cross_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (ffn): Sequential(
    (0): Linear(in_features=512, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
  )
  (norm3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [26]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Tạo một ma trận PE có kích thước (max_len, embed_dim) chứa toàn số 0
        pe = torch.zeros(max_len, embed_dim)
        
        # Tạo một vector cột chứa các vị trí từ 0 đến max_len
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Công thức tính mẫu số theo hàm Sin/Cos của bài báo gốc
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        
        # Gán giá trị Sin cho các cột chẵn (0, 2, 4...)
        pe[:, 0::2] = torch.sin(position * div_term)
        # Gán giá trị Cos cho các cột lẻ (1, 3, 5...)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Thêm chiều Batch vào phía trước: [1, max_len, embed_dim]
        # để lát nữa có thể cộng trực tiếp với Input Tensor
        pe = pe.unsqueeze(0)
        
        # register_buffer giúp lưu ma trận này vào state_dict nhưng KHÔNG học (không cập nhật gradient)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [Batch, Seq_len, embed_dim]
        # Ta chỉ lấy chiều dài đúng bằng Seq_len hiện tại của x để cộng vào
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, dropout=0.1):
        super(TransformerEncoder, self).__init__()
        self.embed_dim = embed_dim
        
        # Lớp chuyển ID từ thành Vector output_vector = embedding_matrix[ID]
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # Lớp thêm thông tin vị trí
        self.pos_encoder = PositionalEncoding(embed_dim=embed_dim, dropout=dropout)
        
        # Xếp chồng N lớp EncoderLayer của bạn lại với nhau
        # nn.ModuleList hoạt động giống hệt một Python List, nhưng nó báo cho PyTorch biết 
        # để theo dõi các tham số bên trong.
        self.layers = nn.ModuleList(
            [EncoderLayer(embed_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)]
        )

    def forward(self, x, src_mask=None):
        # Input x lúc này không là vector nữa
        # Nó chỉ là một ma trận chứa các số nguyên (ID của từ). Shape: [Batch, Seq_len]
        
        # Embedding -> Shape đổi thành: [Batch, Seq_len, embed_dim]
        # Bài báo gốc có nhân thêm căn bậc 2 của embed_dim để scale các giá trị lên
        x = self.embedding(x) * math.sqrt(self.embed_dim)
        
        # Cộng mã hóa vị trí
        x = self.pos_encoder(x)
        
        for layer in self.layers:
            # Nếu bạn có cài đặt mask để che các từ <PAD>, truyền vào đây.
            # Lưu ý: Hàm forward của EncoderLayer bạn viết ở trên đang chưa có tham số mask.
            # Nếu muốn xài mask, bạn cần vào class EncoderLayer sửa lại chút nhé!
            x = layer(x, memory_mask=src_mask)
            
        return x

In [27]:
encoder = TransformerEncoder(vocab_size=10000, embed_dim=512, num_heads=8, ff_dim=2048, num_layers=6)
print(encoder)

TransformerEncoder(
  (embedding): Embedding(10000, 512)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): ReLU()
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)


In [28]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        self.embed_dim = embed_dim
        
        # Lớp chuyển ID từ thành Vector output_vector = embedding_matrix[ID]
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # Lớp thêm thông tin vị trí
        self.pos_encoder = PositionalEncoding(embed_dim=embed_dim, dropout=dropout)
        
        # Xếp chồng N lớp DecoderLayer của bạn lại với nhau
        self.layers = nn.ModuleList(
            [DecoderLayer(embed_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)]
        )

    def forward(self, x, enc_output, tgt_mask=None, memory_mask=None):
        # Input x lúc này không là vector nữa
        # Nó chỉ là một ma trận chứa các số nguyên (ID của từ). Shape: [Batch, Seq_len]
        
        # Embedding -> Shape đổi thành: [Batch, Seq_len, embed_dim]
        x = self.embedding(x) * math.sqrt(self.embed_dim)
        
        # Cộng mã hóa vị trí
        x = self.pos_encoder(x)
        
        for layer in self.layers:
            # mask để che các từ <PAD> hoặc mask causal
            x = layer(x, enc_output, tgt_mask=tgt_mask, memory_mask=memory_mask)
        
        return x

In [29]:
decoder = TransformerDecoder(vocab_size=10000, embed_dim=512, num_heads=8, ff_dim=2048, num_layers=6)
print(decoder)

TransformerDecoder(
  (embedding): Embedding(10000, 512)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layers): ModuleList(
    (0-5): 6 x DecoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (cross_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): ReLU()
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
      (norm3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)


In [30]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_dim, num_heads, ff_dim, num_layers, dropout=0.1):
        super(Seq2SeqTransformer, self).__init__()
        
        # Encoder
        self.encoder = TransformerEncoder(
            src_vocab_size, embed_dim, num_heads, ff_dim, num_layers, dropout
        )
        
        # Decoder
        self.decoder = TransformerDecoder(
            tgt_vocab_size, embed_dim, num_heads, ff_dim, num_layers, dropout
        )
        
        # Generator xuất ra từ vựng
        self.generator = nn.Linear(embed_dim, tgt_vocab_size)

    def forward(self, src, tgt, tgt_mask=None, src_padding_mask=None, tgt_padding_mask=None):
        # Mã hóa câu gốc
        memory = self.encoder(src) # (Có thể thêm src_padding_mask nếu cần)
        
        # Giải mã câu đích (Truyền memory từ Encoder sang)
        output = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=src_padding_mask)
        
        # Bước 3: Dự đoán xác suất cho từ tiếp theo
        logits = self.generator(output)
        
        return logits

In [31]:
trans = Seq2SeqTransformer(src_vocab_size=10000, tgt_vocab_size=10000, embed_dim=512, num_heads=8, ff_dim=2048, num_layers=6)
print(trans)

Seq2SeqTransformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(10000, 512)
    (pos_encoder): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): ReLU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): TransformerDecoder(
    (embedding): Embedding(10000, 512)
    (pos_encoder): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-5):

In [34]:
SRC_VOCAB = 5000   # Kích thước từ điển gốc (Tiếng Anh)
TGT_VOCAB = 8000   # Kích thước từ điển đích (Tiếng Việt)
EMBED_DIM = 256
NUM_HEADS = 8
FF_DIM = 512
NUM_LAYERS = 4
BATCH_SIZE = 2

model = Seq2SeqTransformer(SRC_VOCAB, TGT_VOCAB, EMBED_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS)

# Tạo Dữ Liệu Giả
# Câu gốc (Source) dài 7 từ
src_input = torch.randint(0, SRC_VOCAB, (BATCH_SIZE, 7)) 

# Câu đích (Target) đang dịch dở dang dài 4 từ
tgt_input = torch.randint(0, TGT_VOCAB, (BATCH_SIZE, 4)) 

# Tạo mặt nạ che tương lai (Look-ahead mask) cho câu đích dài 4 từ
tgt_mask = torch.tril(torch.ones(4, 4))
# Đổi 1 -> 0.0, đổi 0 -> -inf cho hàm Additive Mask
tgt_mask = tgt_mask.masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))

# Chạy Mô Hình
print(f"1. Đưa câu gốc vào: {src_input.shape} -> [Batch, Src_Len]")
print(f"2. Đưa câu đích vào: {tgt_input.shape} -> [Batch, Tgt_Len]")

logits = model(src=src_input, tgt=tgt_input, tgt_mask=tgt_mask)

print("-" * 60)
print(f"3. Output dự đoán (Logits): {logits.shape} -> [Batch, Tgt_Len, TGT_VOCAB]")
print(logits)

1. Đưa câu gốc vào: torch.Size([2, 7]) -> [Batch, Src_Len]
2. Đưa câu đích vào: torch.Size([2, 4]) -> [Batch, Tgt_Len]
------------------------------------------------------------
3. Output dự đoán (Logits): torch.Size([2, 4, 8000]) -> [Batch, Tgt_Len, TGT_VOCAB]
tensor([[[-0.3813, -0.1901, -1.1077,  ..., -0.6570,  0.2163,  0.2309],
         [-0.2556, -1.7065, -0.5244,  ..., -0.0259,  0.0941,  0.1172],
         [-0.3957, -0.3069, -0.3527,  ..., -0.5769,  0.2513,  0.1110],
         [-0.1713, -1.7522, -0.3647,  ..., -0.3277, -0.3097,  0.5678]],

        [[-0.6072, -0.7563,  0.8140,  ..., -0.1631, -0.2012,  0.1939],
         [-0.3388, -1.1473,  0.8013,  ..., -0.1864, -0.0741, -0.1751],
         [ 0.2103, -0.0130,  0.2287,  ..., -0.4156, -0.5392, -0.4135],
         [ 0.0431, -0.3047, -0.5028,  ..., -0.0200,  0.1002, -0.2313]]],
       grad_fn=<ViewBackward0>)
